In [102]:
import pandas as pd
import numpy as np
import joblib, json, time, os, warnings
from sklearn.ensemble import IsolationForest
from sklearn.metrics import (precision_score, recall_score, f1_score,
                             classification_report, confusion_matrix,
                             precision_recall_curve)
import shap

In [103]:
warnings.filterwarnings("ignore")
np.random.seed(42)
 
os.makedirs("artifacts", exist_ok=True)
 
print("=" * 60)
print("SentinelIQ — Core ML Pipeline")
print("=" * 60)

SentinelIQ — Core ML Pipeline


In [104]:
# ══════════════════════════════════════════════════════════════
# STEP 1 — LOAD & VALIDATE
# ══════════════════════════════════════════════════════════════
print("\n[STEP 1] Loading fixed dataset...")
 
data_path = "../data/nepal_auth_logins_fixed.csv"
if not os.path.exists(data_path):
    data_path = "./data/nepal_auth_logins_fixed.csv"
df = pd.read_csv(data_path)
y  = (df['label'] == 'anomaly').astype(int)
 
assert df.isnull().sum().sum() == 0,     "FAIL: Nulls found"
assert 'display_risk_score' in df.columns, "FAIL: risk_score not renamed"
assert df[df['label']=='normal']['is_tor'].sum() == 0, "FAIL: Tor in normal rows"
 
print(f"  Rows: {len(df):,}  |  Features: {df.shape[1]}")
print(f"  Normal: {(y==0).sum():,}  |  Anomaly: {(y==1).sum():,}  ({y.mean()*100:.1f}%)")
print(f"  Anomaly types:\n{df[df['label']=='anomaly']['anomaly_type'].value_counts().to_string()}")
print("  All validation checks passed ✓")


[STEP 1] Loading fixed dataset...
  Rows: 19,972  |  Features: 35
  Normal: 18,279  |  Anomaly: 1,693  (8.5%)
  Anomaly types:
anomaly_type
odd_hours_vpn                     420
impossible_travel                 411
credential_stuffing               410
new_device_suspicious_location    362
near_border_foreign_ip             90
  All validation checks passed ✓


In [105]:
# ══════════════════════════════════════════════════════════════
# STEP 2 — FEATURE ENGINEERING
# Derive additional composite features from existing columns
# These amplify signal separation between normal and anomalous
# ══════════════════════════════════════════════════════════════
print("\n[STEP 2] Feature engineering — creating composite signals...")
 
df2 = df.copy()
 
# 2a. velocity_spike_score
# Combines 1hr and 24hr velocity into a single spike signal
# High 1hr with low 24hr = sudden burst = suspicious
df2['velocity_spike_score'] = np.where(
    df2['velocity_last_24hr'] > 0,
    df2['velocity_last_1hr'] / (df2['velocity_last_24hr'] / 24 + 1e-6),
    df2['velocity_last_1hr']
).clip(0, 50)
print("  + velocity_spike_score  (burst detection)")


[STEP 2] Feature engineering — creating composite signals...
  + velocity_spike_score  (burst detection)


In [106]:
# 2b. geo_time_risk
# distance / hours_since_last_login = effective travel speed
# Combines both geo signals into one physics-based score
df2['geo_time_risk'] = np.where(
    df2['hours_since_last_login'] > 0,
    df2['distance_from_last_login_km'] / (df2['hours_since_last_login'] + 0.01),
    df2['distance_from_last_login_km']
).clip(0, 2000)
print("  + geo_time_risk         (travel speed km/h)")
 
# 2c. bot_score
# Low ms_between_attempts + high velocity = bot behavior
# Normalize ms to 0-1 (lower ms = higher bot score)
df2['bot_score'] = (
    (1 - (df2['ms_between_attempts'].clip(0, 30000) / 30000)) * 0.6 +
    (df2['velocity_last_1hr'].clip(0, 30) / 30) * 0.4
).clip(0, 1)
print("  + bot_score             (bot behavior composite)")

  + geo_time_risk         (travel speed km/h)
  + bot_score             (bot behavior composite)


In [107]:
# 2d. trust_deficit_score
# Combines device trust + new device + account age
# New device on young account = high risk
df2['trust_deficit_score'] = (
    df2['device_trust_score'] * 0.5 +
    df2['is_new_device'].astype(int) * 0.3 +
    (1 - (df2['user_account_age_days'].clip(0, 365) / 365)) * 0.2
).clip(0, 1)
print("  + trust_deficit_score   (device + account trust)")

# 2e. network_risk_score
# Combines VPN + datacenter + Tor + ip_reputation
df2['network_risk_score'] = (
    df2['is_known_vpn'].astype(int) * 0.25 +
    df2['is_datacenter_ip'].astype(int) * 0.25 +
    df2['is_tor'].astype(int) * 0.35 +
    df2['ip_reputation'] * 0.15
).clip(0, 1)
print("  + network_risk_score    (IP/network composite)")
 
# Final feature set — base + engineered
FEATURES = [
    # Base signals
    'hour_of_day',
    'hour_deviation',
    'distance_from_last_login_km',
    'hours_since_last_login',
    'impossible_travel',
    'fail_count_before_success',
    'user_account_age_days',
    # Engineered composites
    'velocity_spike_score',
    'geo_time_risk',
    'bot_score',
    'trust_deficit_score',
    'network_risk_score',
]
 
X = df2[FEATURES].copy()
for col in X.select_dtypes(include='bool').columns:
    X[col] = X[col].astype(int)
 
print(f"\n  Total features: {len(FEATURES)} (10 base + 5 engineered composites)")
print(f"  Feature list: {FEATURES}")

  + trust_deficit_score   (device + account trust)
  + network_risk_score    (IP/network composite)

  Total features: 12 (10 base + 5 engineered composites)
  Feature list: ['hour_of_day', 'hour_deviation', 'distance_from_last_login_km', 'hours_since_last_login', 'impossible_travel', 'fail_count_before_success', 'user_account_age_days', 'velocity_spike_score', 'geo_time_risk', 'bot_score', 'trust_deficit_score', 'network_risk_score']


In [108]:
# ══════════════════════════════════════════════════════════════
# STEP 3 — TIME-BASED TRAIN / VAL / TEST SPLIT
# Random split is wrong for time-series login data.
# Always split chronologically — train on past, validate mid, test on future.
# ══════════════════════════════════════════════════════════════
print("\n[STEP 3] Time-based train/val/test split...")
 
df2['timestamp'] = pd.to_datetime(df2['timestamp'], format='mixed')
df2_sorted = df2.sort_values('timestamp').reset_index(drop=True)
X_sorted   = X.loc[df2_sorted.index].reset_index(drop=True)
y_sorted   = y.loc[df2_sorted.index].reset_index(drop=True)
 
n_rows = len(df2_sorted)
train_end = int(n_rows * 0.70)
val_end   = int(n_rows * 0.85)
X_train_all = X_sorted.iloc[:train_end]
X_val       = X_sorted.iloc[train_end:val_end]
X_test      = X_sorted.iloc[val_end:]
y_train_all = y_sorted.iloc[:train_end]
y_val       = y_sorted.iloc[train_end:val_end]
y_test      = y_sorted.iloc[val_end:]


[STEP 3] Time-based train/val/test split...


In [109]:
# Isolation Forest trains on NORMAL rows only — unsupervised
X_train_normal = X_train_all[y_train_all == 0]
 
print(f"  Train period: {df2_sorted['timestamp'].iloc[0].date()} → "
      f"{df2_sorted['timestamp'].iloc[train_end-1].date()}")
print(f"  Val   period: {df2_sorted['timestamp'].iloc[train_end].date()} → "
      f"{df2_sorted['timestamp'].iloc[val_end-1].date()}")
print(f"  Test  period: {df2_sorted['timestamp'].iloc[val_end].date()} → "
      f"{df2_sorted['timestamp'].iloc[-1].date()}")
print(f"  Train rows (normal only): {len(X_train_normal):,}")
print(f"  Val   rows (all):         {len(X_val):,}  "
      f"({y_val.sum()} anomalies = {y_val.mean()*100:.1f}%)")
print(f"  Test  rows (all):         {len(X_test):,}  "
      f"({y_test.sum()} anomalies = {y_test.mean()*100:.1f}%)")

  Train period: 2024-01-01 → 2024-11-28
  Val   period: 2024-11-28 → 2025-03-15
  Test  period: 2025-03-15 → 2025-06-30
  Train rows (normal only): 12,786
  Val   rows (all):         2,996  (235 anomalies = 7.8%)
  Test  rows (all):         2,996  (264 anomalies = 8.8%)


In [110]:
# ══════════════════════════════════════════════════════════════
# STEP 4 — TRAIN ISOLATION FOREST
# Best hyperparams from grid search:
#   n_estimators=300, max_samples=0.8, contamination=0.04
# max_samples=0.8 was the key insight — bootstrapping 80%
# of data per tree increases diversity and reduces false positives
# ══════════════════════════════════════════════════════════════
print("\n[STEP 4] Training Isolation Forest...")
print("  Hyperparams: n_estimators=300, max_samples=0.8, contamination=0.04")
print("  Training on normal rows only (unsupervised)...")
 
t_start = time.perf_counter()
model = IsolationForest(
    n_estimators=300,
    max_samples=0.8,
    contamination=0.04,
    random_state=42,
    n_jobs=-1,
)
model.fit(X_train_normal)
train_time = time.perf_counter() - t_start
print(f"  Training complete in {train_time:.2f}s")


[STEP 4] Training Isolation Forest...
  Hyperparams: n_estimators=300, max_samples=0.8, contamination=0.04
  Training on normal rows only (unsupervised)...
  Training complete in 1.12s


In [111]:
# ══════════════════════════════════════════════════════════════
# STEP 5 — SCORE CALIBRATION
# Raw Isolation Forest scores are negative floats (more negative = more anomalous)
# We normalize to 0.0–1.0 where 1.0 = most anomalous
# This is what the Django API returns as "risk_score"
# ══════════════════════════════════════════════════════════════
print("\n[STEP 5] Calibrating scores to 0.0–1.0 risk scale...")
 
# Get raw scores on TRAINING normal data to establish the normalization range
raw_train_scores = model.score_samples(X_train_normal)
score_min = float(raw_train_scores.min())
score_max = float(raw_train_scores.max())
 
def calibrate(raw_scores, s_min, s_max):
    """Normalize raw IF scores to [0, 1] — 1.0 = highest risk"""
    clipped = np.clip(raw_scores, s_min, s_max)
    normalized = (clipped - s_max) / (s_min - s_max + 1e-10)
    return np.clip(normalized, 0.0, 1.0)
 
# Apply to val/test sets
raw_val   = model.score_samples(X_val)
risk_val  = calibrate(raw_val, score_min, score_max)
raw_test  = model.score_samples(X_test)
risk_test = calibrate(raw_test, score_min, score_max)
 
print(f"  Score range on train normal: [{score_min:.4f}, {score_max:.4f}]")
print(f"  Calibrated val scores:")
print(f"    Normal  rows — mean: {risk_val[y_val==0].mean():.3f}  "
      f"std: {risk_val[y_val==0].std():.3f}  "
      f"max: {risk_val[y_val==0].max():.3f}")
print(f"    Anomaly rows — mean: {risk_val[y_val==1].mean():.3f}  "
      f"std: {risk_val[y_val==1].std():.3f}  "
      f"min: {risk_val[y_val==1].min():.3f}")
 
# Save normalization params for Django
scaler_params = {"score_min": score_min, "score_max": score_max}
with open(os.path.join("artifacts", "scaler_params.json"), "w") as f:
    json.dump(scaler_params, f, indent=2)
print("  Saved: artifacts/scaler_params.json")


[STEP 5] Calibrating scores to 0.0–1.0 risk scale...
  Score range on train normal: [-0.7086, -0.3618]
  Calibrated val scores:
    Normal  rows — mean: 0.183  std: 0.127  max: 0.793
    Anomaly rows — mean: 0.907  std: 0.163  min: 0.226
  Saved: artifacts/scaler_params.json


In [112]:
# ══════════════════════════════════════════════════════════════
# STEP 6 — 4-TIER THRESHOLD MAPPING
# Find thresholds on calibrated scores that maximize:
#   - Tier 1 (allow):    Precision on normal > 99%
#   - Tier 2 (MFA):      Catch early-stage suspicion
#   - Tier 3 (lock):     High confidence anomaly
#   - Tier 4 (block):    Near-certain malicious
# We use Precision-Recall curve on the validation set to avoid test leakage
# ══════════════════════════════════════════════════════════════
print("\n[STEP 6] Finding optimal 4-tier thresholds...")
 
prec_curve, rec_curve, thresh_curve = precision_recall_curve(y_val, risk_val)
 
# Tier 2 threshold: target false-positive rate on validation normals
TARGET_FPR = 0.01
normal_val_scores = risk_val[y_val == 0]
T2 = float(np.quantile(normal_val_scores, 1.0 - TARGET_FPR))
t2_fpr = float((normal_val_scores >= T2).mean())
 
# Tier 3 threshold: where precision >= 0.75
t3_candidates = thresh_curve[prec_curve[:-1] >= 0.75]
T3 = float(t3_candidates.min()) if len(t3_candidates) > 0 else 0.65
 
# Tier 4 threshold: where precision >= 0.90
t4_candidates = thresh_curve[prec_curve[:-1] >= 0.90]
T4 = float(t4_candidates.min()) if len(t4_candidates) > 0 else 0.85
 
# Enforce monotonic thresholds with small gaps
EPS = 0.01
T3 = max(T3, T2 + EPS)
T4 = max(T4, T3 + EPS)
 
print(f"  Tier 2 target FPR: {TARGET_FPR*100:.2f}% (val), achieved: {t2_fpr*100:.2f}%")
print(f"  Tier 1 — ALLOW:      risk < {T2:.2f}")
print(f"  Tier 2 — MFA:        {T2:.2f} <= risk < {T3:.2f}")
print(f"  Tier 3 — LOCK+ALERT: {T3:.2f} <= risk < {T4:.2f}")
print(f"  Tier 4 — HARD BLOCK: risk >= {T4:.2f}")


[STEP 6] Finding optimal 4-tier thresholds...
  Tier 2 target FPR: 1.00% (val), achieved: 1.01%
  Tier 1 — ALLOW:      risk < 0.59
  Tier 2 — MFA:        0.59 <= risk < 0.60
  Tier 3 — LOCK+ALERT: 0.60 <= risk < 0.61
  Tier 4 — HARD BLOCK: risk >= 0.61


In [119]:
def get_tier(risk_score):
    if   risk_score >= T4: return 4, "hard_block",   "CRITICAL"
    elif risk_score >= T3: return 3, "lock_and_alert","HIGH"
    elif risk_score >= T2: return 2, "mfa_required",  "MEDIUM"
    else:                  return 1, "allow",          "LOW"
 
# Post-filter to reduce false positives. This is the sweet spot.
POST_FILTER_MIN_NETWORK = 0.05
POST_FILTER_MIN_GEO = 50
test_df = df2_sorted.iloc[val_end:].reset_index(drop=True)
 
# Tier-2 candidates
tier2_mask = (risk_test >= T2) & (risk_test < T3)
 
# The post-filter is ONLY applied to Tier-2 candidates.
# We are looking for rows that have LOW network and geo risk to demote them.
post_filter_mask = ( 
    (test_df['network_risk_score'] < POST_FILTER_MIN_NETWORK) &
    (test_df['geo_time_risk'] < POST_FILTER_MIN_GEO)
)
 
# Final prediction: start with all scores >= T2
y_pred_t2 = (risk_test >= T2).astype(int)
# Now, demote only the Tier-2 rows that match the filter criteria (low risk)
y_pred_t2[tier2_mask & post_filter_mask] = 0
 
# Predictions for other tiers
y_pred_t3 = (risk_test >= T3).astype(int)
y_pred_t4 = (risk_test >= T4).astype(int)
 
# Evaluate tier distribution on test set
tier_counts = {1:0, 2:0, 3:0, 4:0}
for i in range(len(risk_test)):
    if y_pred_t2[i] == 0:
        tier_counts[1] += 1
    else:
        tier, _, _ = get_tier(risk_test[i])
        tier_counts[tier] += 1
 
print(f"\n  Test set tier distribution (after filtering):")
for t, cnt in tier_counts.items():
    pct = cnt / len(risk_test) * 100
    print(f"    Tier {t}: {cnt:,} rows ({pct:.1f}%)")
 
thresholds = {"T2_mfa": T2, "T3_lock": T3, "T4_block": T4,
              "post_filter": {"network_risk_min": POST_FILTER_MIN_NETWORK,
                                "geo_time_risk_min": POST_FILTER_MIN_GEO}}
with open(os.path.join("artifacts", "thresholds.json"), "w") as f:
    json.dump(thresholds, f, indent=2)
print("  Saved: artifacts/thresholds.json")
 
# --- Detailed Tier Metrics ---
cm_t2 = confusion_matrix(y_test, y_pred_t2)
tn, fp, fn, tp = cm_t2.ravel()
 
print("\n--- Detailed Tier Metrics ---")
print(f"\n[Tier 1: ALLOW]")
print(f"  - Correctly Allowed (True Negatives): {tn:,}")
print(f"  - Incorrectly Allowed (False Negatives): {fn:,}")
print(f"  - False Negative Rate (Miss Rate): {fn / (fn + tp):.3f}")
 
print(f"\n[Tier 2+: MFA / LOCK / BLOCK]")
print(f"  - Precision: {precision_score(y_test, y_pred_t2):.3f}")
print(f"  - Recall:    {recall_score(y_test, y_pred_t2):.3f}")
print(f"  - F1:        {f1_score(y_test, y_pred_t2):.3f}")
 
print(f"\n[Tier 3+: LOCK / BLOCK]")
print(f"  - Precision: {precision_score(y_test, y_pred_t3):.3f}")
print(f"  - Recall:    {recall_score(y_test, y_pred_t3):.3f}")
print(f"  - F1:        {f1_score(y_test, y_pred_t3):.3f}")
 
print(f"\n[Tier 4: BLOCK]")
print(f"  - Precision: {precision_score(y_test, y_pred_t4):.3f}")
print(f"  - Recall:    {recall_score(y_test, y_pred_t4):.3f}")
print(f"  - F1:        {f1_score(y_test, y_pred_t4):.3f}")


  Test set tier distribution (after filtering):
    Tier 1: 2,736 rows (91.3%)
    Tier 2: 0 rows (0.0%)
    Tier 3: 2 rows (0.1%)
    Tier 4: 258 rows (8.6%)
  Saved: artifacts/thresholds.json

--- Detailed Tier Metrics ---

[Tier 1: ALLOW]
  - Correctly Allowed (True Negatives): 2,707
  - Incorrectly Allowed (False Negatives): 29
  - False Negative Rate (Miss Rate): 0.110

[Tier 2+: MFA / LOCK / BLOCK]
  - Precision: 0.904
  - Recall:    0.890
  - F1:        0.897

[Tier 3+: LOCK / BLOCK]
  - Precision: 0.904
  - Recall:    0.890
  - F1:        0.897

[Tier 4: BLOCK]
  - Precision: 0.911
  - Recall:    0.890
  - F1:        0.900


In [ ]:
# --- Tier distribution: validation vs test ---
def tier_counts_for(scores, apply_tier2_filter):
    if apply_tier2_filter:
        tier2_mask_local = (scores >= T2) & (scores < T3)
        post_filter_local = (
            (df2_sorted.iloc[val_end:].reset_index(drop=True)['network_risk_score'] < POST_FILTER_MIN_NETWORK) &
            (df2_sorted.iloc[val_end:].reset_index(drop=True)['geo_time_risk'] < POST_FILTER_MIN_GEO)
        )
        y_pred_local = (scores >= T2).astype(int)
        y_pred_local[tier2_mask_local & post_filter_local] = 0
    else:
        y_pred_local = (scores >= T2).astype(int)
 
    counts = {1: 0, 2: 0, 3: 0, 4: 0}
    for i in range(len(scores)):
        if y_pred_local[i] == 0:
            counts[1] += 1
        else:
            t, _, _ = get_tier(scores[i])
            counts[t] += 1
    return counts
 
val_counts = tier_counts_for(risk_val, apply_tier2_filter=False)
test_counts = tier_counts_for(risk_test, apply_tier2_filter=True)
 
print("\n--- Tier Distribution (Val vs Test) ---")
for tier in [1, 2, 3, 4]:
    v = val_counts[tier]
    t = test_counts[tier]
    v_pct = v / len(risk_val) * 100
    t_pct = t / len(risk_test) * 100
    print(f"Tier {tier}: Val={v:,} ({v_pct:.1f}%) | Test={t:,} ({t_pct:.1f}%)")
 
# --- Risk score overlap check ---
print("\n--- Risk Score Summary (Val vs Test) ---")
print(f"Val normal mean={risk_val[y_val==0].mean():.3f}, std={risk_val[y_val==0].std():.3f}")
print(f"Val anomaly mean={risk_val[y_val==1].mean():.3f}, std={risk_val[y_val==1].std():.3f}")
print(f"Test normal mean={risk_test[y_test==0].mean():.3f}, std={risk_test[y_test==0].std():.3f}")
print(f"Test anomaly mean={risk_test[y_test==1].mean():.3f}, std={risk_test[y_test==1].std():.3f}")

In [114]:
# ══════════════════════════════════════════════════════════════
# STEP 7 — SHAP EXPLAINABILITY
# TreeExplainer is the right choice for Isolation Forest.
# We pre-initialize it at startup so inference is fast.
# ══════════════════════════════════════════════════════════════
print("\n[STEP 7] Building SHAP explainer + demo explanations...")
 
explainer = shap.TreeExplainer(model)
 
# Demo explanation for each anomaly type
print("\n  --- SHAP explanations per anomaly type ---")
df2_sorted_reset = df2_sorted.reset_index(drop=True)
X_sorted_reset   = X_sorted.reset_index(drop=True)
y_sorted_reset   = y_sorted.reset_index(drop=True)
 
for atype in ['impossible_travel','credential_stuffing',
              'odd_hours_vpn','new_device_suspicious_location',
              'near_border_foreign_ip']:
    atype_mask = df2_sorted_reset['anomaly_type'] == atype
    if not atype_mask.any():
        continue
    idx    = atype_mask.idxmax()
    sample = X_sorted_reset.iloc[[idx]]
    sv     = -explainer.shap_values(sample)
    shap_df = pd.DataFrame({
        'feature': FEATURES,
        'shap':    sv[0]
    }).sort_values('shap', ascending=False)
 
    print(f"\n  [{atype}]")
    for _, row in shap_df.head(3).iterrows():
        direction = "RAISES risk" if row['shap'] > 0 else "lowers risk"
        print(f"    {row['feature']:<30} SHAP={row['shap']:+.4f}  ({direction})")


[STEP 7] Building SHAP explainer + demo explanations...

  --- SHAP explanations per anomaly type ---

  [impossible_travel]
    geo_time_risk                  SHAP=+6.3126  (RAISES risk)
    network_risk_score             SHAP=+3.8237  (RAISES risk)
    hour_of_day                    SHAP=+1.5476  (RAISES risk)

  [credential_stuffing]
    network_risk_score             SHAP=+0.3447  (RAISES risk)
    fail_count_before_success      SHAP=+0.3054  (RAISES risk)
    hour_of_day                    SHAP=+0.0884  (RAISES risk)

  [odd_hours_vpn]
    velocity_spike_score           SHAP=+2.1971  (RAISES risk)
    distance_from_last_login_km    SHAP=+0.4680  (RAISES risk)
    impossible_travel              SHAP=-0.0000  (lowers risk)

  [new_device_suspicious_location]
    distance_from_last_login_km    SHAP=+0.8000  (RAISES risk)
    hours_since_last_login         SHAP=+0.5021  (RAISES risk)
    user_account_age_days          SHAP=+0.3804  (RAISES risk)

  [near_border_foreign_ip]
    distan

In [115]:
# ══════════════════════════════════════════════════════════════
# STEP 8 — PER ANOMALY-TYPE EVALUATION
# ══════════════════════════════════════════════════════════════
print("\n[STEP 8] Per-anomaly-type evaluation on test set...")
 
test_df = df2_sorted.iloc[val_end:].reset_index(drop=True)
test_df['risk_score']  = risk_test
test_df['y_true']      = y_test.values
test_df['y_pred_t2']   = y_pred_t2
test_df['tier'] = [get_tier(s)[0] for s in risk_test]
 
metrics_by_type = {}
print(f"\n  {'Anomaly Type':<42} {'N':>5}  {'Recall':>7}  {'Avg Score':>10}")
print(f"  {'-'*42} {'-'*5}  {'-'*7}  {'-'*10}")
 
for atype in test_df[test_df['label']=='anomaly']['anomaly_type'].unique():
    sub = test_df[test_df['anomaly_type'] == atype]
    if len(sub) == 0:
        continue
    rec       = recall_score(sub['y_true'], sub['y_pred_t2'], zero_division=0)
    avg_score = sub['risk_score'].mean()
    print(f"  {atype:<42} {len(sub):>5}  {rec:>7.3f}  {avg_score:>10.3f}")
    metrics_by_type[atype] = {"n": len(sub), "recall": round(rec,3),
                              "avg_risk_score": round(avg_score,3)}
 
# Normal rows false positive rate
normal_test = test_df[test_df['label']=='normal']
fpr = (normal_test['y_pred_t2'] == 1).mean()
print(f"\n  Normal rows flagged (false positive rate): {fpr*100:.2f}%")
print(f"  That means 1 in {int(1/fpr) if fpr > 0 else 'inf'} normal logins gets a false alert")


[STEP 8] Per-anomaly-type evaluation on test set...

  Anomaly Type                                   N   Recall   Avg Score
  ------------------------------------------ -----  -------  ----------
  near_border_foreign_ip                        56    1.000       0.224
  new_device_suspicious_location                49    1.000       0.250
  credential_stuffing                           63    0.800       0.248
  odd_hours_vpn                                 58    0.750       0.271
  impossible_travel                             59    1.000       0.303

  Normal rows flagged (false positive rate): 8.52%
  That means 1 in 11 normal logins gets a false alert


In [116]:
# ══════════════════════════════════════════════════════════════
# STEP 9 — LATENCY BENCHMARK
# This is what you quote to judges and mentors
# ══════════════════════════════════════════════════════════════
print("\n[STEP 9] Latency benchmark (1000 iterations)...")
 
sample_row = X_test.iloc[[0]]
sv_times, if_times, calib_times = [], [], []
 
for _ in range(1000):
    # IF prediction
    t0 = time.perf_counter()
    raw = model.score_samples(sample_row)
    if_times.append((time.perf_counter()-t0)*1000)
 
    # Calibration
    t0 = time.perf_counter()
    calibrate(raw, score_min, score_max)
    calib_times.append((time.perf_counter()-t0)*1000)
 
# SHAP (50 iterations — it's slower)
for _ in range(50):
    t0 = time.perf_counter()
    explainer.shap_values(sample_row)
    sv_times.append((time.perf_counter()-t0)*1000)
 
print(f"\n  {'Component':<30} {'Mean':>8}  {'P50':>8}  {'P95':>8}  {'P99':>8}")
print(f"  {'-'*30} {'-'*8}  {'-'*8}  {'-'*8}  {'-'*8}")
for name, times in [
    ("Isolation Forest predict", if_times),
    ("Score calibration",        calib_times),
    ("SHAP explanation",         sv_times),
]:
    t = np.array(times)
    print(f"  {name:<30} {t.mean():>7.2f}ms  "
          f"{np.percentile(t,50):>7.2f}ms  "
          f"{np.percentile(t,95):>7.2f}ms  "
          f"{np.percentile(t,99):>7.2f}ms")
 
total_no_shap = np.mean(if_times) + np.mean(calib_times)
total_w_shap  = total_no_shap + np.mean(sv_times)
print(f"\n  Total (no SHAP — normal logins):    {total_no_shap:.2f}ms")
print(f"  Total (with SHAP — flagged logins): {total_w_shap:.2f}ms")
print(f"  Note: SHAP only fires for risk >= {T3:.2f} (Tier 3+)")


[STEP 9] Latency benchmark (1000 iterations)...

  Component                          Mean       P50       P95       P99
  ------------------------------ --------  --------  --------  --------
  Isolation Forest predict         29.41ms    28.01ms    41.47ms    49.69ms
  Score calibration                 0.03ms     0.02ms     0.04ms     0.08ms
  SHAP explanation                146.97ms   145.35ms   164.39ms   186.54ms

  Total (no SHAP — normal logins):    29.44ms
  Total (with SHAP — flagged logins): 176.41ms
  Note: SHAP only fires for risk >= 0.60 (Tier 3+)


In [117]:
# ══════════════════════════════════════════════════════════════
# STEP 10 — SAVE ALL ARTIFACTS
# ══════════════════════════════════════════════════════════════
print("\n[STEP 10] Saving artifacts for Django...")
 
# Model + explainer
joblib.dump(model,    os.path.join("artifacts", "model.pkl"))
joblib.dump(explainer,os.path.join("artifacts", "explainer.pkl"))
 
# Feature importance (mean |SHAP| on test anomalies)
anom_test_X = X_test[y_test==1]
if len(anom_test_X) > 0:
    sv_all = explainer.shap_values(anom_test_X.iloc[:200])
    fi     = pd.DataFrame({
        "feature":    FEATURES,
        "importance": np.abs(sv_all).mean(axis=0)
    }).sort_values("importance", ascending=False)
else:
    fi = pd.DataFrame({"feature": FEATURES, "importance": [0]*len(FEATURES)})
 
fi_dict = fi.set_index("feature")["importance"].round(4).to_dict()
with open(os.path.join("artifacts", "feature_importance.json"),"w") as f:
    json.dump(fi_dict, f, indent=2)
 
print("\n  Feature importance (mean |SHAP| on anomalies):")
for feat, imp in list(fi_dict.items())[:8]:
    bar = "█" * int(imp * 30)
    print(f"    {feat:<30} {imp:.4f}  {bar}")
 
# Final metrics JSON
final_metrics = {
    "model":           "IsolationForest",
    "n_estimators":    300,
    "max_samples":     0.8,
    "contamination":   0.04,
    "n_features":      len(FEATURES),
    "features":        FEATURES,
    "train_rows":      len(X_train_normal),
    "test_rows":       len(X_test),
    "test_anomalies":  int(y_test.sum()),
    "precision":       round(precision_score(y_test, y_pred_t2), 4),
    "recall":          round(recall_score(y_test, y_pred_t2), 4),
    "f1":              round(f1_score(y_test, y_pred_t2), 4),
    "false_positive_rate": round(float(fpr), 4),
    "thresholds":      thresholds,
    "latency_ms": {
        "if_predict_mean":   round(float(np.mean(if_times)),   2),
        "if_predict_p99":    round(float(np.percentile(if_times,99)),2),
        "shap_mean":         round(float(np.mean(sv_times)),   2),
        "shap_p99":          round(float(np.percentile(sv_times,99)),2),
        "total_normal_ms":   round(total_no_shap,  2),
        "total_flagged_ms":  round(total_w_shap,   2),
    },
    "per_type_recall": metrics_by_type,
}
 
with open(os.path.join("artifacts", "metrics.json"),"w") as f:
    json.dump(final_metrics, f, indent=2)
 
print("\n  Saved artifacts:")
for fname in ["model.pkl","explainer.pkl","scaler_params.json",
              "thresholds.json","metrics.json","feature_importance.json"]:
    size = os.path.getsize(os.path.join("artifacts", fname))
    print(f"    artifacts/{fname:<30} ({size:,} bytes)")


[STEP 10] Saving artifacts for Django...

  Feature importance (mean |SHAP| on anomalies):
    geo_time_risk                  5.8478  ███████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████
    network_risk_score             2.2769  ████████████████████████████████████████████████████████████████████
    distance_from_last_login_km    1.2985  ██████████████████████████████████████
    hour_of_day                    0.7849  ███████████████████████
    trust_deficit_score            0.7434  ██████████████████████
    fail_count_before_success      0.5071  ███████████████
    bot_score                      0.4628  █████████████
    hour_deviation                 0.4218  ████████████

  Saved artifacts:
    artifacts/model.pkl                      (65,926,729 bytes)
    artifacts/explainer.pkl                  (93,493,806 bytes)
    artifacts/scaler_params.json          

In [118]:
# ══════════════════════════════════════════════════════════════
# FINAL SUMMARY
# ══════════════════════════════════════════════════════════════
print("\n" + "="*60)
print("PIPELINE COMPLETE — RESULTS SUMMARY")
print("="*60)
print(f"""
Dataset
  Total rows:      {len(df2):,}
  Train (normal):  {len(X_train_normal):,}
  Test (all):      {len(X_test):,}
 
Model
  Algorithm:       Isolation Forest
  n_estimators:    300
  max_samples:     0.8  ← key hyperparameter
  contamination:   0.04
  Features:        {len(FEATURES)} (base + engineered composites)
 
Metrics (test set, Tier-2+ threshold)
  Precision:       {final_metrics['precision']*100:.1f}%
  Recall:          {final_metrics['recall']*100:.1f}%
  F1:              {final_metrics['f1']*100:.1f}%
  False Pos Rate:  {final_metrics['false_positive_rate']*100:.2f}%
 
Latency
  Normal login:    {total_no_shap:.2f}ms  (no SHAP)
  Flagged login:   {total_w_shap:.2f}ms   (with SHAP)
 
Thresholds
  Tier 1 ALLOW:    risk < {T2:.2f}
  Tier 2 MFA:      {T2:.2f} – {T3:.2f}
  Tier 3 LOCK:     {T3:.2f} – {T4:.2f}
  Tier 4 BLOCK:    >= {T4:.2f}
 
How to load in Django:
  import joblib, json
  model     = joblib.load('ml/artifacts/model.pkl')
  explainer = joblib.load('ml/artifacts/explainer.pkl')
  params    = json.load(open('ml/artifacts/scaler_params.json'))
  thresholds= json.load(open('ml/artifacts/thresholds.json'))
""")


PIPELINE COMPLETE — RESULTS SUMMARY

Dataset
  Total rows:      19,972
  Train (normal):  12,786
  Test (all):      2,996

Model
  Algorithm:       Isolation Forest
  n_estimators:    300
  max_samples:     0.8  ← key hyperparameter
  contamination:   0.04
  Features:        12 (base + engineered composites)

Metrics (test set, Tier-2+ threshold)
  Precision:       90.4%
  Recall:          89.0%
  F1:              89.7%
  False Pos Rate:  8.52%

Latency
  Normal login:    29.44ms  (no SHAP)
  Flagged login:   176.41ms   (with SHAP)

Thresholds
  Tier 1 ALLOW:    risk < 0.59
  Tier 2 MFA:      0.59 – 0.60
  Tier 3 LOCK:     0.60 – 0.61
  Tier 4 BLOCK:    >= 0.61

How to load in Django:
  import joblib, json
  model     = joblib.load('ml/artifacts/model.pkl')
  explainer = joblib.load('ml/artifacts/explainer.pkl')
  params    = json.load(open('ml/artifacts/scaler_params.json'))
  thresholds= json.load(open('ml/artifacts/thresholds.json'))

